# Tutorial: Pandas Functions Mastery Guide

Audience:
- Learners who already know basic Python but want to become confident with real pandas workflows.
- Anyone who has used pandas a little and still wonders "when do I use `concat` instead of `merge`?" or "what exactly does `groupby` return?"

Prerequisites:
- Python variables, functions, and lists/dictionaries.
- Basic tabular thinking: rows, columns, keys, and aggregates.

Learning goals:
- Build deterministic random datasets for practice.
- Understand the mental model behind selection, filtering, sorting, and assignment.
- Know when to use `concat`, `merge`, `join`, `melt`, `pivot`, and `pivot_table`.
- See what `groupby` really returns and how `agg`, `transform`, and `filter` differ.
- Work with multi-level indexes and multi-level column headers without fear.



## Outline

1. Generate realistic practice data
2. Inspect and clean DataFrames
3. Choose between `concat` and `merge`
4. Understand `groupby` objects and outputs
5. Create and manage multi-level headers
6. Reshape tables with `melt`, `pivot`, and `stack`
7. Finish with exercises and solution scaffolds

The notebook uses a single mini business scenario so every function feels connected.



In [1]:
from __future__ import annotations

# If pandas is not installed yet in your notebook kernel, uncomment one of these lines:
# %pip install pandas numpy
# !pip install pandas numpy

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

SEED = 7
rng = np.random.default_rng(SEED)

print("pandas version:", pd.__version__)
print("Seed:", SEED)



pandas version: 3.0.1
Seed: 7


## Step 1 - Build deterministic random datasets

We create synthetic but realistic tables:
- `customers`: one row per customer
- `orders`: multiple rows per customer
- `monthly_targets`: department-level planning data
- `quarterly_sales_wide`: a wide table for reshaping practice

The important point is reproducibility: because the random seed is fixed, your results stay stable.



In [2]:
n_customers = 18
customer_ids = np.arange(1001, 1001 + n_customers)

customers = pd.DataFrame(
{
"customer_id": customer_ids,
"segment": rng.choice(["Consumer", "Corporate", "Home Office"], size=n_customers, p=[0.5, 0.3, 0.2]),
"country": rng.choice(["France", "Germany", "Switzerland"], size=n_customers, p=[0.4, 0.35, 0.25]),
"signup_channel": rng.choice(["Web", "Partner", "Sales"], size=n_customers),
"loyalty_score": rng.integers(40, 100, size=n_customers),
}
).sort_values("customer_id").reset_index(drop=True)

n_orders = 90
orders = pd.DataFrame(
{
"order_id": np.arange(5001, 5001 + n_orders),
"customer_id": rng.choice(customer_ids, size=n_orders, replace=True),
"order_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 120, size=n_orders), unit="D"),
"product_family": rng.choice(["Analytics", "Automation", "Storage"], size=n_orders, p=[0.4, 0.35, 0.25]),
"region": rng.choice(["North", "South", "East", "West"], size=n_orders),
"quantity": rng.integers(1, 8, size=n_orders),
"unit_price": rng.choice([49, 79, 99, 149, 199], size=n_orders),
"discount_rate": rng.choice([0.0, 0.05, 0.10, 0.15], size=n_orders, p=[0.45, 0.25, 0.2, 0.1]),
}
)

orders["gross_revenue"] = orders["quantity"] * orders["unit_price"]
orders["net_revenue"] = orders["gross_revenue"] * (1 - orders["discount_rate"])
orders["order_month"] = orders["order_date"].dt.to_period("M").astype(str)

month_index = pd.period_range("2025-01", "2025-04", freq="M").astype(str)
region_list = ["North", "South", "East", "West"]
target_grid = pd.MultiIndex.from_product(
[month_index, region_list],
names=["order_month", "region"],
)
monthly_targets = target_grid.to_frame(index=False)
monthly_targets["target_revenue"] = rng.integers(2600, 3800, size=len(monthly_targets))

quarterly_sales_wide = pd.DataFrame(
{
"region": ["North", "South", "East", "West"],
"Q1": rng.integers(2200, 4000, size=4),
"Q2": rng.integers(2200, 4000, size=4),
"Q3": rng.integers(2200, 4000, size=4),
"Q4": rng.integers(2200, 4000, size=4),
}
)

customers.head()



,customer_id,segment,country,signup_channel,loyalty_score
0,1001,Corporate,Germany,Sales,42
1,1002,Home Office,Switzerland,Partner,84
2,1003,Corporate,France,Partner,68
3,1004,Consumer,France,Web,45
4,1005,Consumer,Germany,Partner,54


In [3]:
orders.head()



,order_id,customer_id,order_date,product_family,region,quantity,unit_price,discount_rate,gross_revenue,net_revenue,order_month
0,5001,1011,2025-04-19,Automation,North,4,79,0.00,316,316.00,2025-04
1,5002,1006,2025-01-15,Storage,North,6,199,0.00,1194,"1,194.00",2025-01
2,5003,1005,2025-02-19,Analytics,South,6,79,0.00,474,474.00,2025-02
3,5004,1003,2025-04-27,Automation,North,5,49,0.15,245,208.25,2025-04
4,5005,1007,2025-01-14,Automation,South,3,99,0.15,297,252.45,2025-01


In [4]:
target_grid



MultiIndex([('2025-01', 'North'),
            ('2025-01', 'South'),
            ('2025-01',  'East'),
            ('2025-01',  'West'),
            ('2025-02', 'North'),
            ('2025-02', 'South'),
            ('2025-02',  'East'),
            ('2025-02',  'West'),
            ('2025-03', 'North'),
            ('2025-03', 'South'),
            ('2025-03',  'East'),
            ('2025-03',  'West'),
            ('2025-04', 'North'),
            ('2025-04', 'South'),
            ('2025-04',  'East'),
            ('2025-04',  'West')],
           names=['order_month', 'region'])

## Step 2 - Inspect the structure before doing anything fancy

Beginners often jump directly into `merge` or `groupby`, but pandas becomes much easier when you inspect:
- the shape
- the dtypes
- the meaning of the index
- the presence of missing values or duplicates



In [5]:
print("customers shape:", customers.shape)
print("orders shape:", orders.shape)

orders.info()



customers shape: (18, 5)
orders shape: (90, 11)
<class 'pandas.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        90 non-null     int64         
 1   customer_id     90 non-null     int64         
 2   order_date      90 non-null     datetime64[us]
 3   product_family  90 non-null     str           
 4   region          90 non-null     str           
 5   quantity        90 non-null     int64         
 6   unit_price      90 non-null     int64         
 7   discount_rate   90 non-null     float64       
 8   gross_revenue   90 non-null     int64         
 9   net_revenue     90 non-null     float64       
 10  order_month     90 non-null     str           
dtypes: datetime64[us](1), float64(2), int64(5), str(3)
memory usage: 7.9 KB


In [6]:
orders.describe(include="all").T[["count", "unique", "mean", "min", "max"]]



,count,unique,mean,min,max
order_id,90.00,NaN,"5,045.50","5,001.00","5,090.00"
customer_id,90.00,NaN,"1,009.44","1,001.00","1,018.00"
order_date,90,NaN,2025-03-08 07:44:00,2025-01-01 00:00:00,2025-04-30 00:00:00
product_family,90,3,NaN,NaN,NaN
region,90,4,NaN,NaN,NaN
quantity,90.00,NaN,4.11,1.00,7.00
unit_price,90.00,NaN,113.00,49.00,199.00
discount_rate,90.00,NaN,0.06,0.00,0.15
gross_revenue,90.00,NaN,454.11,49.00,"1,393.00"
net_revenue,90.00,NaN,426.44,46.55,"1,253.70"


In [7]:
# A quick quality check is often worth doing before analysis.
quality_checks = pd.DataFrame(
{
"missing_values": orders.isna().sum(),
"dtype": orders.dtypes.astype(str),
"n_unique": orders.nunique(),
}
)

quality_checks



,missing_values,dtype,n_unique
order_id,0,int64,90
customer_id,0,int64,18
order_date,0,datetime64[us],64
product_family,0,str,3
region,0,str,4
quantity,0,int64,7
unit_price,0,int64,5
discount_rate,0,float64,4
gross_revenue,0,int64,35
net_revenue,0,float64,65


## Step 3 - Common row and column operations

This section is less glamorous, but these are the moves you use every day:
- selecting columns
- filtering rows with boolean masks or `query`
- sorting
- creating derived columns with `assign`
- using `loc` when you want both row and column logic to stay explicit



In [8]:
selected = orders[["order_id", "customer_id", "region", "net_revenue"]].head(8)

high_value = orders.loc[
(orders["net_revenue"] > 500) & (orders["discount_rate"] <= 0.10),
["order_id", "region", "product_family", "net_revenue"],
].sort_values("net_revenue", ascending=False)

january_analytics = orders.query("order_month == '2025-01' and product_family == 'Analytics'")

selected, high_value.head(5), january_analytics.head(5)



(   order_id  customer_id region  net_revenue
 0      5001         1011  North       316.00
 1      5002         1006  North     1,194.00
 2      5003         1005  South       474.00
 3      5004         1003  North       208.25
 4      5005         1007  South       252.45
 5      5006         1015   East       147.00
 6      5007         1008   East       759.90
 7      5008         1007  North       199.00,
     order_id region product_family  net_revenue
 24      5025   West      Analytics     1,253.70
 21      5022  North     Automation     1,253.70
 1       5002  North        Storage     1,194.00
 16      5017   West      Analytics     1,134.30
 78      5079  North     Automation     1,074.60,
     order_id  customer_id order_date product_family region  quantity  unit_price  discount_rate  gross_revenue  \
 14      5015         1009 2025-01-07      Analytics  North         2         199           0.00            398   
 43      5044         1016 2025-01-02      Analytics  South 

In [9]:
# assign returns a new DataFrame, which is handy in method chains
orders_enriched = (
orders
.assign(
revenue_band=lambda df: pd.cut(
df["net_revenue"],
bins=[0, 200, 500, 1000, np.inf],
labels=["Small", "Medium", "Large", "Enterprise"],
right=False,
),
is_discounted=lambda df: df["discount_rate"] > 0,
)
)

orders_enriched[["order_id", "net_revenue", "revenue_band", "is_discounted"]].head(10)



,order_id,net_revenue,revenue_band,is_discounted
0,5001,316.00,Medium,False
1,5002,"1,194.00",Enterprise,False
2,5003,474.00,Medium,False
3,5004,208.25,Medium,True
4,5005,252.45,Medium,True
5,5006,147.00,Small,False
6,5007,759.90,Large,True
7,5008,199.00,Small,False
8,5009,93.10,Small,True
9,5010,186.20,Small,True


## Step 4 - `concat` versus `merge`: the mental model

This is one of the most important pandas distinctions.

Use `pd.concat(...)` when you want to glue objects together along an axis:
- stack rows from similar tables
- place columns side by side using the index

Use `DataFrame.merge(...)` when you want SQL-style matching based on key columns:
- enrich orders with customer attributes
- combine facts with lookup tables

Short rule:
- `concat` is about position and axis.
- `merge` is about key relationships.



In [10]:
jan_orders = orders.query("order_month == '2025-01'").copy()
feb_orders = orders.query("order_month == '2025-02'").copy()

stacked_orders = pd.concat([jan_orders, feb_orders], axis=0, ignore_index=True)

print("January rows:", len(jan_orders))
print("February rows:", len(feb_orders))
print("After vertical concat:", len(stacked_orders))

stacked_orders[["order_id", "order_month", "customer_id"]].head()



January rows: 20
February rows: 18
After vertical concat: 38


,order_id,order_month,customer_id
0,5002,2025-01,1006
1,5005,2025-01,1007
2,5009,2025-01,1011
3,5015,2025-01,1009
4,5028,2025-01,1002


In [11]:
# Horizontal concat aligns on the index, not on key columns.
customer_profile = customers.set_index("customer_id")[["segment", "country"]]
customer_scores = customers.set_index("customer_id")[["loyalty_score"]]

side_by_side = pd.concat([customer_profile, customer_scores], axis=1)
side_by_side.head()



,segment,country,loyalty_score
customer_id,,,
1001,Corporate,Germany,42
1002,Home Office,Switzerland,84
1003,Corporate,France,68
1004,Consumer,France,45
1005,Consumer,Germany,54


In [12]:
# merge matches rows by key columns. This is what you want for relational enrichment.
orders_with_customers = orders.merge(
customers,
on="customer_id",
how="left",
validate="many_to_one",
indicator=True,
)

orders_with_customers[["order_id", "customer_id", "segment", "country", "_merge"]].head()



,order_id,customer_id,segment,country,_merge
0,5001,1011,Consumer,Germany,both
1,5002,1006,Home Office,France,both
2,5003,1005,Consumer,Germany,both
3,5004,1003,Corporate,France,both
4,5005,1007,Consumer,France,both


### Why `merge` is not the same as horizontal `concat`

If two DataFrames share the same key values but not the same index values, horizontal `concat` can silently misalign rows.
`merge` prevents that by asking: "which rows match on this key?"



In [13]:
wrong_horizontal_concat = pd.concat(
[
orders[["order_id", "customer_id"]].head(5).reset_index(drop=True),
customers[["customer_id", "country"]].head(5).reset_index(drop=True),
],
axis=1,
)

correct_merge = (
orders[["order_id", "customer_id"]]
.head(5)
.merge(customers[["customer_id", "country"]], on="customer_id", how="left")
)

print("Horizontal concat can look valid while being logically wrong.")
wrong_horizontal_concat, correct_merge



Horizontal concat can look valid while being logically wrong.


(   order_id  customer_id  customer_id      country
 0      5001         1011         1001      Germany
 1      5002         1006         1002  Switzerland
 2      5003         1005         1003       France
 3      5004         1003         1004       France
 4      5005         1007         1005      Germany,
    order_id  customer_id  country
 0      5001         1011  Germany
 1      5002         1006   France
 2      5003         1005  Germany
 3      5004         1003   France
 4      5005         1007   France)

## Step 5 - Important `merge` parameters

Parameters you will use constantly:
- `on`: shared key column(s)
- `left_on` / `right_on`: when the key names differ
- `how`: `left`, `right`, `inner`, `outer`
- `suffixes`: disambiguate overlapping column names
- `indicator=True`: show where rows came from
- `validate=...`: catch accidental row explosions



In [14]:
customer_lookup = customers.rename(columns={"customer_id": "cust_id"})

merged_with_different_key_names = orders.merge(
customer_lookup[["cust_id", "segment", "country"]],
left_on="customer_id",
right_on="cust_id",
how="left",
validate="many_to_one",
)

merged_with_different_key_names[["order_id", "customer_id", "cust_id", "segment"]].head()



,order_id,customer_id,cust_id,segment
0,5001,1011,1011,Consumer
1,5002,1006,1006,Home Office
2,5003,1005,1005,Consumer
3,5004,1003,1003,Corporate
4,5005,1007,1007,Consumer


In [15]:
# A tiny example to visualize the four join styles clearly.
left = pd.DataFrame({"key": ["A", "B", "C"], "left_value": [10, 20, 30]})
right = pd.DataFrame({"key": ["B", "C", "D"], "right_value": [200, 300, 400]})

join_examples = {
join_type: left.merge(right, on="key", how=join_type, indicator=True)
for join_type in ["inner", "left", "right", "outer"]
}

join_examples["inner"], join_examples["outer"], join_examples["left"], join_examples["right"]



(  key  left_value  right_value _merge
 0   B          20          200   both
 1   C          30          300   both,
   key  left_value  right_value      _merge
 0   A       10.00          NaN   left_only
 1   B       20.00       200.00        both
 2   C       30.00       300.00        both
 3   D         NaN       400.00  right_only,
   key  left_value  right_value     _merge
 0   A          10          NaN  left_only
 1   B          20       200.00       both
 2   C          30       300.00       both,
   key  left_value  right_value      _merge
 0   B       20.00          200        both
 1   C       30.00          300        both
 2   D         NaN          400  right_only)

In [16]:
# validate is a safety belt. Here we create a duplicate key on purpose.
duplicate_customer_lookup = pd.concat(
[
customers[["customer_id", "segment"]],
customers[["customer_id", "segment"]].iloc[[0]],
],
ignore_index=True,
)

try:
    orders.merge(
    duplicate_customer_lookup,
    on="customer_id",
    how="left",
    validate="many_to_one",
    )
except Exception as exc:
    print(type(exc).__name__)
    print(exc)



MergeError
Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
  customer_id
        1001 ...


## Step 6 - What does `groupby` actually return?

`groupby` does **not** immediately compute a final table.
It returns a lazy grouping object that remembers:
- the original data
- the grouping keys
- how rows are partitioned

You then ask that object for something:
- `.size()` for counts
- `.sum()` or `.mean()` for simple reductions
- `.agg(...)` for custom summary tables
- `.transform(...)` to return one value per original row
- `.filter(...)` to keep or discard full groups



In [17]:
grouped = orders.groupby("region")

print(type(grouped))
print("Group names:", list(grouped.groups.keys()))
print("Row indexes in North group:", grouped.groups["North"][:5])



<class 'pandas.api.typing.DataFrameGroupBy'>
Group names: ['East', 'North', 'South', 'West']
Row indexes in North group: Index([0, 1, 3, 7, 8], dtype='int64')


In [18]:
region_counts = grouped.size().rename("order_count")
region_revenue = grouped["net_revenue"].sum().rename("total_net_revenue")

region_counts, region_revenue



(region
 East     23
 North    25
 South    21
 West     21
 Name: order_count, dtype: int64,
 region
 East     9,397.45
 North   10,648.15
 South    7,919.05
 West    10,415.40
 Name: total_net_revenue, dtype: float64)

In [19]:
grouped_summary = (
orders
.groupby(["region", "product_family"], as_index=False)
.agg(
orders_count=("order_id", "count"),
avg_quantity=("quantity", "mean"),
total_revenue=("net_revenue", "sum"),
)
.sort_values(["region", "total_revenue"], ascending=[True, False])
)

grouped_summary.head(10)



,region,product_family,orders_count,avg_quantity,total_revenue
1,East,Automation,11,5.18,"4,993.25"
0,East,Analytics,7,4.57,"3,422.05"
2,East,Storage,5,3.00,982.15
5,North,Storage,11,4.09,"5,185.95"
4,North,Automation,8,4.25,"3,728.30"
3,North,Analytics,6,2.83,"1,733.90"
6,South,Analytics,9,4.22,"3,802.30"
8,South,Storage,6,3.67,"2,107.30"
7,South,Automation,6,4.83,"2,009.45"
9,West,Analytics,11,4.00,"6,125.25"


In [20]:
# transform returns a Series aligned with the original rows.
# That makes it perfect for percentages, z-scores, and within-group normalization.
orders_with_share = orders.assign(
region_total=lambda df: df.groupby("region")["net_revenue"].transform("sum"),
pct_of_region=lambda df: df["net_revenue"] / df["region_total"],
)

orders_with_share[["order_id", "region", "net_revenue", "region_total", "pct_of_region"]].head()



,order_id,region,net_revenue,region_total,pct_of_region
0,5001,North,316.00,"10,648.15",0.03
1,5002,North,"1,194.00","10,648.15",0.11
2,5003,South,474.00,"7,919.05",0.06
3,5004,North,208.25,"10,648.15",0.02
4,5005,South,252.45,"7,919.05",0.03


In [21]:
# filter keeps full groups when the condition is true for the group.
regions_with_large_average = orders.groupby("region").filter(
lambda group: group["net_revenue"].mean() > 350
)

regions_with_large_average[["order_id", "region", "net_revenue"]].head()



,order_id,region,net_revenue
0,5001,North,316.00
1,5002,North,"1,194.00"
2,5003,South,474.00
3,5004,North,208.25
4,5005,South,252.45


### `as_index=False` versus default behavior

Many people get confused because grouped columns become an index by default.

- Default: grouping keys become the index in the result.
- `as_index=False`: grouping keys stay as normal columns.

Both are valid. Choose the shape that makes the next step easiest.



In [22]:
default_grouped = orders.groupby("region")["net_revenue"].sum()
flat_grouped = orders.groupby("region", as_index=False)["net_revenue"].sum()

print("Default result type:", type(default_grouped))
print("Flat result type:", type(flat_grouped))

default_grouped, flat_grouped



Default result type: <class 'pandas.Series'>
Flat result type: <class 'pandas.DataFrame'>


(region
 East     9,397.45
 North   10,648.15
 South    7,919.05
 West    10,415.40
 Name: net_revenue, dtype: float64,
   region  net_revenue
 0   East     9,397.45
 1  North    10,648.15
 2  South     7,919.05
 3   West    10,415.40)

## Step 7 - Multi-level column headers from `agg`

Multi-level headers often appear after:
- `groupby(...).agg({...})`
- `pivot_table(...)`
- some reshaping operations

They are not scary; they are just tuples of labels.



In [23]:
multi_header_summary = orders.groupby("region").agg(
{
"quantity": ["sum", "mean", "max"],
"net_revenue": ["sum", "mean", "max"],
}
)

multi_header_summary



quantity          net_revenue                
            sum mean max         sum   mean      max
region                                              
East        104 4.52   7    9,397.45 408.58 1,043.00
North        96 3.84   7   10,648.15 425.93 1,253.70
South        89 4.24   7    7,919.05 377.10   849.30
West         81 3.86   7   10,415.40 495.97 1,253.70

In [24]:
print("Column index object:", type(multi_header_summary.columns))
print("Column labels as tuples:", list(multi_header_summary.columns))

multi_header_summary[("net_revenue", "mean")]



Column index object: <class 'pandas.MultiIndex'>
Column labels as tuples: [('quantity', 'sum'), ('quantity', 'mean'), ('quantity', 'max'), ('net_revenue', 'sum'), ('net_revenue', 'mean'), ('net_revenue', 'max')]


region
East    408.58
North   425.93
South   377.10
West    495.97
Name: (net_revenue, mean), dtype: float64

In [25]:
# Flattening is common when you want easy downstream access or CSV export.
flattened = multi_header_summary.copy()
flattened.columns = ["_".join(col) for col in flattened.columns]
flattened = flattened.reset_index()

flattened.head()



,region,quantity_sum,quantity_mean,quantity_max,net_revenue_sum,net_revenue_mean,net_revenue_max
0,East,104,4.52,7,"9,397.45",408.58,"1,043.00"
1,North,96,3.84,7,"10,648.15",425.93,"1,253.70"
2,South,89,4.24,7,"7,919.05",377.10,849.30
3,West,81,3.86,7,"10,415.40",495.97,"1,253.70"


## Step 8 - MultiIndex rows and columns with pivot tables

`pivot_table` is a great bridge between raw transactional data and analysis tables.
It can create:
- multi-level row indexes
- multi-level column headers
- aggregated values in one step



In [26]:
revenue_pivot = pd.pivot_table(
orders,
index=["region", "product_family"],
columns="order_month",
values="net_revenue",
aggfunc="sum",
fill_value=0,
)

revenue_pivot.head(10)



order_month            2025-01  2025-02  2025-03  2025-04
region product_family                                    
East   Analytics          0.00   311.15   759.90 2,351.00
       Automation     1,060.60 1,315.30 2,171.85   445.50
       Storage            0.00   670.00   147.00   165.15
North  Analytics      1,071.30   264.60   398.00     0.00
       Automation        93.10   733.65 1,253.70 1,647.85
       Storage        2,646.15   199.00   150.10 2,190.70
South  Analytics        845.75 1,563.65   411.30   981.60
       Automation       546.45   279.30   450.30   733.40
       Storage          199.00     0.00 1,355.30   553.00
West   Analytics        478.25 1,134.30 1,101.45 3,411.25

In [27]:
print("Row index type:", type(revenue_pivot.index))
print("First row index label:", revenue_pivot.index[0])
print("Column labels:", list(revenue_pivot.columns))

revenue_pivot.loc[("North", "Analytics")]



Row index type: <class 'pandas.MultiIndex'>
First row index label: ('East', 'Analytics')
Column labels: ['2025-01', '2025-02', '2025-03', '2025-04']


order_month
2025-01   1,071.30
2025-02     264.60
2025-03     398.00
2025-04       0.00
Name: (North, Analytics), dtype: float64

In [28]:
# stack moves columns into the index; unstack does the reverse.
pivot_stacked = revenue_pivot.stack().rename("revenue").reset_index()
pivot_stacked.head(8)



,region,product_family,order_month,revenue
0,East,Analytics,2025-01,0.00
1,East,Analytics,2025-02,311.15
2,East,Analytics,2025-03,759.90
3,East,Analytics,2025-04,"2,351.00"
4,East,Automation,2025-01,"1,060.60"
5,East,Automation,2025-02,"1,315.30"
6,East,Automation,2025-03,"2,171.85"
7,East,Automation,2025-04,445.50


## Step 9 - `melt`, `pivot`, and `pivot_table`

Use:
- `melt` to go from wide to long
- `pivot` to go from long to wide when each combination is unique
- `pivot_table` when duplicates exist and you need aggregation



In [29]:
quarterly_sales_long = quarterly_sales_wide.melt(
id_vars="region",
var_name="quarter",
value_name="sales",
)

quarterly_sales_long.head(8)



,region,quarter,sales
0,North,Q1,2359
1,South,Q1,3613
2,East,Q1,3932
3,West,Q1,3005
4,North,Q2,3557
5,South,Q2,3483
6,East,Q2,3813
7,West,Q2,2261


In [30]:
pivoted_back = quarterly_sales_long.pivot(index="region", columns="quarter", values="sales")
pivoted_back



quarter,Q1,Q2,Q3,Q4
region,,,,
East,3932,3813,3746,3088
North,2359,3557,3469,2715
South,3613,3483,2901,3243
West,3005,2261,3748,3204


In [31]:
duplicated_long = pd.concat(
[
quarterly_sales_long,
quarterly_sales_long.iloc[[0]].assign(sales=quarterly_sales_long.iloc[0]["sales"] + 125),
],
ignore_index=True,
)

try:
    duplicated_long.pivot(index="region", columns="quarter", values="sales")
except Exception as exc:
    print(type(exc).__name__)
    print(exc)

# pivot_table works because it can aggregate duplicates.
duplicated_long.pivot_table(index="region", columns="quarter", values="sales", aggfunc="mean")



ValueError
Index contains duplicate entries, cannot reshape


quarter,Q1,Q2,Q3,Q4
region,,,,
East,"3,932.00","3,813.00","3,746.00","3,088.00"
North,"2,421.50","3,557.00","3,469.00","2,715.00"
South,"3,613.00","3,483.00","2,901.00","3,243.00"
West,"3,005.00","2,261.00","3,748.00","3,204.00"


## Step 10 - Time-aware grouping and resampling-style thinking

Even without using full time-series APIs, pandas gets much nicer when you extract time parts explicitly.



In [32]:
monthly_region_summary = (
orders
.groupby(["order_month", "region"], as_index=False)
.agg(
total_orders=("order_id", "count"),
total_revenue=("net_revenue", "sum"),
avg_discount=("discount_rate", "mean"),
)
.sort_values(["order_month", "total_revenue"], ascending=[True, False])
)

monthly_region_summary.head(12)



,order_month,region,total_orders,total_revenue,avg_discount
1,2025-01,North,9,"3,810.55",0.05
2,2025-01,South,4,"1,591.20",0.07
0,2025-01,East,2,"1,060.60",0.05
3,2025-01,West,5,919.25,0.03
4,2025-02,East,7,"2,296.45",0.05
6,2025-02,South,5,"1,842.95",0.04
7,2025-02,West,2,"1,701.45",0.05
5,2025-02,North,4,"1,197.25",0.04
8,2025-03,East,7,"3,078.75",0.07
10,2025-03,South,7,"2,216.90",0.09


In [33]:
# This merge compares actual revenue to the simple target table.
# Because the target table is keyed by month and region, this is a clean many-to-one merge.
actual_vs_target = monthly_region_summary.merge(
monthly_targets,
on=["order_month", "region"],
how="left",
validate="many_to_one",
)

actual_vs_target["gap_to_target_template"] = (
actual_vs_target["total_revenue"] - actual_vs_target["target_revenue"]
)

actual_vs_target.head(8)



,order_month,region,total_orders,total_revenue,avg_discount,target_revenue,gap_to_target_template
0,2025-01,North,9,"3,810.55",0.05,3707,103.55
1,2025-01,South,4,"1,591.20",0.07,3056,"-1,464.80"
2,2025-01,East,2,"1,060.60",0.05,3108,"-2,047.40"
3,2025-01,West,5,919.25,0.03,3159,"-2,239.75"
4,2025-02,East,7,"2,296.45",0.05,3657,"-1,360.55"
5,2025-02,South,5,"1,842.95",0.04,3506,"-1,663.05"
6,2025-02,West,2,"1,701.45",0.05,3204,"-1,502.55"
7,2025-02,North,4,"1,197.25",0.04,2686,"-1,488.75"


## Step 11 - Missing values, duplicates, and safe habits

Real data is messy, so here are a few reliable habits:
- check `isna().sum()`
- check duplicate keys with `.duplicated(...)`
- use `validate=` in merges
- prefer explicit column selection over "I hope this works"



In [34]:
messy_orders = orders.copy()
messy_orders.loc[messy_orders.sample(3, random_state=SEED).index, "discount_rate"] = np.nan
messy_orders = pd.concat([messy_orders, messy_orders.iloc[[0]]], ignore_index=True)

pd.DataFrame(
{
"missing_values": messy_orders.isna().sum(),
"duplicate_order_ids": [messy_orders["order_id"].duplicated().sum()] * len(messy_orders.columns),
},
index=messy_orders.columns,
)



,missing_values,duplicate_order_ids
order_id,0,1
customer_id,0,1
order_date,0,1
product_family,0,1
region,0,1
quantity,0,1
unit_price,0,1
discount_rate,3,1
gross_revenue,0,1
net_revenue,0,1


In [35]:
cleaned_orders = (
messy_orders
.drop_duplicates(subset="order_id")
.assign(discount_rate=lambda df: df["discount_rate"].fillna(0))
)

len(messy_orders), len(cleaned_orders), cleaned_orders["discount_rate"].isna().sum()



(91, 90, np.int64(0))

## Step 12 - A small end-to-end pipeline

This kind of pipeline is where pandas starts to feel elegant:
1. merge lookup data
2. create derived columns
3. group and aggregate
4. sort for reporting



In [36]:
final_report = (
orders
.merge(customers[["customer_id", "segment", "country"]], on="customer_id", how="left")
.assign(
discount_flag=lambda df: np.where(df["discount_rate"] > 0, "Discounted", "Full price"),
month=lambda df: df["order_date"].dt.to_period("M").astype(str),
)
.groupby(["month", "segment", "discount_flag"], as_index=False)
.agg(
orders_count=("order_id", "count"),
customers_count=("customer_id", "nunique"),
revenue=("net_revenue", "sum"),
)
.sort_values(["month", "revenue"], ascending=[True, False])
)

final_report.head(12)



,month,segment,discount_flag,orders_count,customers_count,revenue
2,2025-01,Corporate,Discounted,4,2,"2,026.30"
5,2025-01,Home Office,Full price,3,2,"1,488.00"
0,2025-01,Consumer,Discounted,6,4,"1,305.70"
3,2025-01,Corporate,Full price,2,2,"1,194.00"
1,2025-01,Consumer,Full price,4,2,"1,103.00"
4,2025-01,Home Office,Discounted,1,1,264.60
6,2025-02,Consumer,Discounted,7,3,"3,638.50"
7,2025-02,Consumer,Full price,3,3,"1,267.00"
9,2025-02,Corporate,Full price,2,2,670.00
8,2025-02,Corporate,Discounted,3,3,575.75


## Common mistakes to remember

- Use `concat` when stacking or aligning objects by axis; use `merge` when matching records by key.
- `groupby` returns a grouping object first, not a finished table.
- `transform` returns one value per original row; `agg` returns one row per group.
- `pivot` fails on duplicate combinations; `pivot_table` can aggregate them.
- Multi-level headers are just structured labels. You can select them with tuples or flatten them.



## Final takeaway

Pandas gets much easier when you choose the function based on the table shape you want:
- same rows, new summary values -> `groupby(...).agg(...)`
- same rows, group-based derived value -> `groupby(...).transform(...)`
- combine by key -> `merge`
- stack objects -> `concat`
- wide to long -> `melt`
- long to wide -> `pivot` or `pivot_table`

If you want, the next good exercise is to replace the synthetic data with one of your own CSV files and repeat the same steps.

